[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# executemany &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/stations.db` with the two tables, loads the five stations and the
year of readings as the notebook did, and defines `ids`, `INSERT_READING`, `ROWS`, `TABLES` and
`timed_load`. Run it first. The tasks do not depend on one another, and the last cell closes the
connection and removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
import time
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
TIMING = SCRATCH / "timing.db"
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}
TABLES = """
    CREATE TABLE stations (
        id       INTEGER PRIMARY KEY,
        name     TEXT NOT NULL UNIQUE,
        latitude REAL NOT NULL CONSTRAINT plausible_latitude CHECK (latitude BETWEEN -90 AND 90)
    ) STRICT;
    CREATE TABLE readings (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL REFERENCES stations (id),
        hour       TEXT NOT NULL,
        celsius    REAL CONSTRAINT plausible_celsius CHECK (celsius BETWEEN -90 AND 60),
        UNIQUE (station_id, hour)
    ) STRICT;
"""
INSERT_READING = "INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)"


def year_of_readings():
    """Every hour of 2025 at the four stations, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


def timed_load(rows, *, autocommit, many):
    """Seconds taken to insert rows into new tables in timing.db, with executemany or a loop of execute."""
    TIMING.unlink(missing_ok=True)
    timing = sqlite3.connect(TIMING, autocommit=autocommit)
    timing.executescript(TABLES)
    started = time.perf_counter()
    if many:
        timing.executemany(INSERT_READING, rows)
    else:
        for row in rows:
            timing.execute(INSERT_READING, row)
    timing.commit()
    seconds = time.perf_counter() - started
    timing.close()
    return seconds


conn = sqlite3.connect(DATABASE)
conn.executescript(TABLES)
with conn:
    conn.executemany("INSERT INTO stations (name, latitude) VALUES (?, ?)", LATITUDES.items())
ids = dict(conn.execute("SELECT name, id FROM stations ORDER BY id"))
ROWS = [(ids[station], hour, celsius) for station, hour, celsius in year_of_readings()]
with conn:
    conn.executemany(INSERT_READING, ROWS)

print("stations:", ids, "| readings:", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0])


stations: {'Bergen': 1, 'Oslo': 2, 'Svalbard': 3, 'Tromso': 4, 'Kirkenes': 5} | readings: 35040


**1.** Three stations, from dictionaries.


In [2]:
new_stations = [
    {"name": "Bodo", "latitude": 67.28},
    {"name": "Alta", "latitude": 69.97},
    {"name": "Vardo", "latitude": 70.37},
]
with conn:
    cursor = conn.executemany("INSERT INTO stations (name, latitude) VALUES (:name, :latitude)", new_stations)
print("rowcount:", cursor.rowcount)

names = [station["name"] for station in new_stations]
print(dict(conn.execute("SELECT name, id FROM stations WHERE name IN (?, ?, ?) ORDER BY id", names)))


rowcount: 3
{'Bodo': 6, 'Alta': 7, 'Vardo': 8}


Every dictionary supplied `:name` and `:latitude` by name, and `rowcount` counted the three inserts.
The ids came from reading the stations back by their unique names, since `executemany` does not set
`lastrowid`.


**2.** Tromso's December, a day at a time.


In [3]:
days = [(ids["Tromso"], f"2025-12-{day:02d}") for day in range(1, 32)]
with conn:
    cursor = conn.executemany("DELETE FROM readings WHERE station_id = ? AND date(hour) = ?", days)
print("rowcount:", cursor.rowcount)


rowcount: 744


31 runs of the `DELETE`, one for every day, and `rowcount` added them up: 24 readings a day for 31
days, 744 in all.


**3.** A generator function for a day of readings.


In [4]:
def svalbard_new_year():
    """Svalbard's 24 readings for 1 January 2026, all -15.0."""
    for hour in range(24):
        yield ids["Svalbard"], f"2026-01-01T{hour:02d}:00", -15.0


with conn:
    cursor = conn.executemany(INSERT_READING, svalbard_new_year())
print("rowcount:", cursor.rowcount, "| lastrowid:", cursor.lastrowid)


rowcount: 24 | lastrowid: None


`executemany` asked the generator for one row at a time, and inserted 24. `lastrowid` is `None`,
because `conn.executemany` made a new cursor and `executemany` never sets it.


**4.** Ids from RETURNING, used for readings.


In [5]:
with conn:
    new_ids = {name: conn.execute("INSERT INTO stations (name, latitude) VALUES (?, ?) RETURNING id",
                                  (name, latitude)).fetchone()[0]
               for name, latitude in [("Hammerfest", 70.66), ("Narvik", 68.44)]}
    conn.executemany(INSERT_READING, [(new_ids["Hammerfest"], "2026-01-01T00:00", -6.5),
                                      (new_ids["Narvik"], "2026-01-01T00:00", -3.0)])
print(new_ids)
print(conn.execute("""
    SELECT stations.name, readings.celsius FROM readings JOIN stations ON stations.id = readings.station_id
    WHERE stations.name IN ('Hammerfest', 'Narvik') ORDER BY stations.name
""").fetchall())


{'Hammerfest': 9, 'Narvik': 10}
[('Hammerfest', -6.5), ('Narvik', -3.0)]


`execute` with `RETURNING` gave back each station's id as it was inserted, and the dictionary put the
right id in each reading. The stations and the readings share one transaction, so a failure in the
readings would have removed the stations too.


**5.** 500 rows, timed.


In [6]:
every_row = timed_load(ROWS[:500], autocommit=True, many=True)
one_commit = timed_load(ROWS[:500], autocommit=False, many=True)
print("a commit for every row took more than 20 times as long as one commit:", every_row > 20 * one_commit)


a commit for every row took more than 20 times as long as one commit: True


With `autocommit=True`, each of the 500 inserts was saved on its own and waited for the disk, while
the other load waited once, at its `commit`. The margin in the comparison leaves room for a faster or
slower machine.


**6.** Corrections that all apply, or none.


In [7]:
def apply_corrections(conn, corrections):
    """Apply (station_id, hour, celsius) corrections in one transaction, or none if any matches no reading."""
    with conn:
        cursor = conn.executemany("UPDATE readings SET celsius = ? WHERE station_id = ? AND hour = ?",
                                  [(celsius, station_id, hour) for station_id, hour, celsius in corrections])
        if cursor.rowcount != len(corrections):
            raise ValueError(f"{len(corrections) - cursor.rowcount} of {len(corrections)} corrections matched no reading")
    return cursor.rowcount


noon = "SELECT celsius FROM readings WHERE station_id = ? AND hour = '2025-07-01T12:00'"
print("Oslo at noon on 1 July, before:", conn.execute(noon, (ids["Oslo"],)).fetchone()[0])
try:
    apply_corrections(conn, [(ids["Oslo"], "2025-07-01T12:00", 18.4), (ids["Oslo"], "2025-07-01T25:00", 18.9)])
except ValueError as error:
    print("rolled back:", error)
print("Oslo at noon on 1 July, after: ", conn.execute(noon, (ids["Oslo"],)).fetchone()[0])


Oslo at noon on 1 July, before: 17.5
rolled back: 1 of 2 corrections matched no reading
Oslo at noon on 1 July, after:  17.5


The first correction matched noon, and the second named an hour 25 that no reading has, so `rowcount`
was 1 for 2 corrections. The `ValueError` left the `with` block, which rolled back the first
correction too, and Oslo's noon reading is what it was before.

Last, close the connection and remove the scratch folder:


In [8]:
conn.close()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [executemany](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/14-executemany.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
